In [1]:
import os, shutil, gc, glob
os.environ['PROJ_LIB'] = '/home/jovyan/edskywalker/imber-sen3/settings/sen3_env/share/proj'

import planetary_computer
import pystac_client
import fsspec

from getpass import getpass

import xarray as xr
import numpy as np
import cf_xarray, rioxarray

import time
from datetime import datetime, timedelta

from cdo import Cdo
#cdopath ='/home/jovyan/edskywalker/imber-sen3/settings/sen3_env/bin/cdo'
cdo = Cdo()
cdo.cleanTempDir()

from matplotlib import pyplot as plt
from matplotlib import colors
from matplotlib import cm
import colormaps as cmo

from cartopy import crs as ccrs
from cartopy import feature as cf

import zipfile

#from rich.jupyter import print as rprint
from rich.table import Table
from rich.markdown import Markdown
from rich.console import Console
console = Console()

import warnings
warnings.filterwarnings('ignore')

from tqdm.auto import tqdm

In [2]:
cdo.debug = True

In [3]:
west = 106.588355
east = 107.123103
north = -5.841392
south = -6.163255

area_of_interest = {
    "type": "Polygon",
    "coordinates": [
        [
            [west, south],
            [east, south],
            [east, north],
            [west, north],
            [west, south],
        ]
    ],
}


extent = [west, east, south, north]
bbox = [west, south, east, north]

bbox_str = f'{west},{east},{south},{north}' 

In [4]:
dtstart = '2020-01-01'
dtend = '2020-03-31'

time_of_interest = f"{dtstart}/{dtend}"


In [5]:
catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1", modifier=planetary_computer.sign_inplace)


In [6]:
search = catalog.search(collections=["sentinel-3-slstr-wst-l2-netcdf"], intersects=area_of_interest, datetime=time_of_interest)
items = search.item_collection()

In [7]:
display(items[0])

<Item id=S3B_SL_2_WST_20200331T143847_20200331T161947_6059_037_167>

In [8]:
dataset = xr.open_dataset(fsspec.open(items[0].assets["l2p"].href).open())


In [ ]:
dataset = cdo.sellonlatbox(bbox_str, input = dataset, returnXDataset = True)
display(dataset)

Found operator:sellonlatbox


In [ ]:
SST = dataset["sea_surface_temperature"]
BIAS = dataset["sses_bias"]
ALGORITHM = dataset["sst_algorithm_type"]
QUAL = dataset["quality_level"]

mask = (ALGORITHM >= 4) | (QUAL >= 5)
final_sst = xr.where(mask, SST + BIAS, np.nan)

display(final_sst)